# Stage 01 - Feature Window Search

Route: `lst_models` V2
Scope: `validation_only`
User-facing notebook: `notebooks/01_feature_window_search_colab.ipynb`

This notebook screens predeclared feature sets and window sizes with
lightweight train-inner probes. Stage 01 does not determine the final
model. It produces candidate inputs and a model-family shortlist for
Stage 02 train-inner HPO.


## Protocol Summary

Stage 01 consumes frozen Stage 00 artifacts and must preserve the
Stage 00 label, split, baseline, and no-contact boundary contract.

Allowed search axes:

- `feature_set`
- `window_size` in `[10, 20, 30]`
- lightweight probe rows inside train-inner folds

Forbidden in Stage 01:

- final model claims
- formal HPO
- label, horizon, no-trade band, or split changes
- official validation, test, or holdout use for selection


## Expected Artifacts

Stage 01 writes a compact run folder when `RUN_STAGE01=True`:

```text
results/01_feature_window_search/<run_id>/
  run_manifest.json
  artifact_inventory.csv
  01_feature_window_search_summary.csv
  01_candidate_inputs.json
  01_train_inner_probe_ledger.csv
  01_train_inner_fold_manifest.csv
```

`01_candidate_inputs.json` must record `no_final_model_selected=true`
and `holdout_test_contact=false`.


In [ ]:
from pathlib import Path
import importlib
import json
import subprocess
import sys
import zipfile

RUN_PROJECT_BOOTSTRAP = True
PROJECT_BOOTSTRAP_MODE = "github_commit"  # github_commit | drive_bundle | manual_upload | already_present

PROJECT_REPO_URL = "https://github.com/zkc768/lstm-zhang.git"
PROJECT_REPO_COMMIT = "cf6ced19a0eed10f2e9ed2f210d2e639bcfcb2bc"
PROJECT_ROOT = Path("/content/lst_models")
PROJECT_DRIVE_BUNDLE_FILE_ID = ""
PROJECT_DRIVE_BUNDLE_NAME = "lst_models_stage01_colab_project.zip"

RUN_STAGE00_DRIVE_SYNC = True
RUN_STAGE01 = False
RUN_ARTIFACT_DISPLAY = False

STAGE_NAME = "01_feature_window_search"
ROUTE = "lst_models"
SCOPE = "validation_only"
HOLDOUT_TEST_CONTACT = False

EXPECTED_WINDOWS = [10, 20, 30]
STAGE02_RECOMMENDED_FAMILIES = [
    "lightgbm",
    "dlinear_only",
    "tcn_only",
    "ms_dlinear_tcn",
]
OPTIONAL_FIXED_CONTROLS = ["simple_gru", "shallow_lstm"]


def run_cmd(args, cwd=None):
    print("+", " ".join(str(arg) for arg in args))
    subprocess.run(args, cwd=cwd, check=True)


def looks_like_project_root(path):
    return (
        (path / "configs" / "stages" / "01_feature_window_search.yaml").exists()
        and (path / "docs" / "protocols" / "01_feature_window_search_protocol.md").exists()
        and (path / "notebooks" / "01_feature_window_search_colab.ipynb").exists()
        and (path / "src" / "lst_models").exists()
    )


def safe_extract_project_zip(zip_path):
    destination = Path("/content").resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            member_path = Path(member.filename)
            if member_path.is_absolute() or ".." in member_path.parts:
                raise ValueError(f"Unsafe path in uploaded zip: {member.filename}")
            target = (destination / member_path).resolve()
            if target != destination and destination not in target.parents:
                raise ValueError(f"Unsafe path in uploaded zip: {member.filename}")
        archive.extractall(destination)
    candidates = [Path("/content/lst_models"), Path("/content") / zip_path.stem]
    for candidate in candidates:
        if looks_like_project_root(candidate):
            return candidate
    raise FileNotFoundError(
        "The project bundle was extracted, but no lst_models project root was found. "
        "The bundle must contain configs/, docs/, notebooks/, and src/."
    )


def download_and_extract_project_zip_from_drive():
    try:
        from google.colab import auth
        from googleapiclient.discovery import build
        from googleapiclient.http import MediaIoBaseDownload
    except ImportError as exc:
        raise RuntimeError(
            "PROJECT_BOOTSTRAP_MODE='drive_bundle' only works inside Colab "
            "with Google Drive API dependencies available."
        ) from exc

    file_id = PROJECT_DRIVE_BUNDLE_FILE_ID.strip()
    if not file_id:
        raise ValueError(
            "PROJECT_DRIVE_BUNDLE_FILE_ID is required when "
            "PROJECT_BOOTSTRAP_MODE='drive_bundle'."
        )
    auth.authenticate_user()
    service = build("drive", "v3")
    zip_path = Path("/content") / PROJECT_DRIVE_BUNDLE_NAME
    request = service.files().get_media(fileId=file_id)
    with zip_path.open("wb") as file_handle:
        downloader = MediaIoBaseDownload(file_handle, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()
    return safe_extract_project_zip(zip_path)


def upload_and_extract_project_zip():
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError(
            "PROJECT_BOOTSTRAP_MODE='manual_upload' only works inside Colab."
        ) from exc

    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("No project zip was uploaded.")
    zip_names = [name for name in uploaded if name.endswith(".zip")]
    if not zip_names:
        raise ValueError("Upload a .zip file containing the lst_models project folder.")
    zip_path = Path("/content") / zip_names[0]
    return safe_extract_project_zip(zip_path)


if RUN_PROJECT_BOOTSTRAP:
    if PROJECT_BOOTSTRAP_MODE == "github_commit":
        if (PROJECT_ROOT / ".git").exists():
            run_cmd(["git", "fetch", "origin"], cwd=PROJECT_ROOT)
            run_cmd(["git", "checkout", PROJECT_REPO_COMMIT], cwd=PROJECT_ROOT)
        else:
            run_cmd(["git", "clone", PROJECT_REPO_URL, str(PROJECT_ROOT)])
            run_cmd(["git", "checkout", PROJECT_REPO_COMMIT], cwd=PROJECT_ROOT)
        actual_commit = subprocess.check_output(
            ["git", "rev-parse", "HEAD"],
            cwd=PROJECT_ROOT,
            text=True,
        ).strip()
        assert actual_commit == PROJECT_REPO_COMMIT, (actual_commit, PROJECT_REPO_COMMIT)
    elif PROJECT_BOOTSTRAP_MODE == "drive_bundle":
        PROJECT_ROOT = download_and_extract_project_zip_from_drive()
    elif PROJECT_BOOTSTRAP_MODE == "manual_upload":
        PROJECT_ROOT = upload_and_extract_project_zip()
    elif PROJECT_BOOTSTRAP_MODE == "already_present":
        pass
    else:
        raise ValueError(
            "PROJECT_BOOTSTRAP_MODE must be one of: "
            "github_commit, drive_bundle, manual_upload, already_present"
        )

STAGE_CONFIG_PATH = PROJECT_ROOT / "configs" / "stages" / "01_feature_window_search.yaml"
PROTOCOL_PATH = PROJECT_ROOT / "docs" / "protocols" / "01_feature_window_search_protocol.md"
NOTEBOOK_PATH = PROJECT_ROOT / "notebooks" / "01_feature_window_search_colab.ipynb"
STAGE_ENTRYPOINT_PATH = PROJECT_ROOT / "src" / "lst_models" / "stages" / "feature_window_search.py"
STAGE00_RUN_ID = "20260608_164408"
STAGE00_OUTPUT_DIR = Path("/content/lst_models_results/00_data_split_label_freeze") / STAGE00_RUN_ID
STAGE00_DRIVE_PATH_PARTS = ["lst_models", "results", "00_data_split_label_freeze", STAGE00_RUN_ID]
OUTPUT_DIR = Path("/content/lst_models_results/01_feature_window_search")

REQUIRED_PROJECT_FILES = [
    STAGE_CONFIG_PATH,
    PROTOCOL_PATH,
    NOTEBOOK_PATH,
    PROJECT_ROOT / "src" / "lst_models",
]
missing_project_files = [path for path in REQUIRED_PROJECT_FILES if not path.exists()]
if missing_project_files:
    missing_text = "\n".join(f"- {path}" for path in missing_project_files)
    raise FileNotFoundError(
        "Stage 01 bootstrap target is missing required package sidecars. "
        "For normal Colab use, push the full stage bundle to GitHub and update "
        "PROJECT_REPO_COMMIT to that exact commit. Drive bundle/manual upload "
        "are fallback modes only. "
        f"Missing files:\n{missing_text}"
    )

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

def clear_project_import_cache():
    cached = [name for name in sys.modules if name == "lst_models" or name.startswith("lst_models.")]
    for name in cached:
        del sys.modules[name]
    importlib.invalidate_caches()

clear_project_import_cache()

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROJECT_BOOTSTRAP_MODE:", PROJECT_BOOTSTRAP_MODE)
print("PROJECT_REPO_URL:", PROJECT_REPO_URL)
print("PROJECT_COMMIT:", PROJECT_REPO_COMMIT)
print("SRC_PATH:", SRC_PATH)
print("STAGE_CONFIG_PATH:", STAGE_CONFIG_PATH)
print("PROTOCOL_PATH:", PROTOCOL_PATH)
print("NOTEBOOK_PATH:", NOTEBOOK_PATH)
print("STAGE_ENTRYPOINT_PATH:", STAGE_ENTRYPOINT_PATH)
print("STAGE00_RUN_ID:", STAGE00_RUN_ID)
print("STAGE00_OUTPUT_DIR:", STAGE00_OUTPUT_DIR)
print("STAGE00_DRIVE_PATH_PARTS:", STAGE00_DRIVE_PATH_PARTS)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("RUN_PROJECT_BOOTSTRAP:", RUN_PROJECT_BOOTSTRAP)
print("RUN_STAGE00_DRIVE_SYNC:", RUN_STAGE00_DRIVE_SYNC)
print("RUN_STAGE01:", RUN_STAGE01)
print("RUN_ARTIFACT_DISPLAY:", RUN_ARTIFACT_DISPLAY)


## Config Load And Contract Check

This cell reads the Stage 01 config sidecar and checks the
notebook-facing contract. It does not read market data or model
metrics.


In [ ]:
try:
    import yaml
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "PyYAML is required to read the Stage 01 config. "
        "Install project dependencies before running this notebook."
    ) from exc


def require_path(path):
    if not path.exists():
        raise FileNotFoundError(f"missing required Stage 01 file: {path}")


require_path(STAGE_CONFIG_PATH)
require_path(PROTOCOL_PATH)

with STAGE_CONFIG_PATH.open("r", encoding="utf-8") as handle:
    stage01_config = yaml.safe_load(handle)

stage00_inputs = stage01_config["inputs"]
assert stage01_config["stage_name"] == STAGE_NAME
assert stage01_config["route"] == ROUTE
assert stage01_config["scope"] == SCOPE
assert stage01_config["holdout_test_contact"] is HOLDOUT_TEST_CONTACT
assert stage01_config["window_sizes"] == EXPECTED_WINDOWS
assert stage01_config["selection_rules"]["no_final_model_selected"] is True
assert stage00_inputs["stage00_run_id"] == STAGE00_RUN_ID
assert Path(stage00_inputs["stage00_runtime_run_dir"]) == STAGE00_OUTPUT_DIR
assert stage00_inputs["stage00_drive_path_parts"] == STAGE00_DRIVE_PATH_PARTS
assert (
    stage01_config["stage02_handoff"]["recommended_model_families"]
    == STAGE02_RECOMMENDED_FAMILIES
)
assert stage01_config["optional_fixed_controls"]["simple_gru"]["stage02_hpo_family"] is False
assert stage01_config["optional_fixed_controls"]["shallow_lstm"]["stage02_hpo_family"] is False

print(json.dumps({
    "stage_name": stage01_config["stage_name"],
    "scope": stage01_config["scope"],
    "window_sizes": stage01_config["window_sizes"],
    "source_stage00_run_id": stage00_inputs["stage00_run_id"],
    "stage00_runtime_run_dir": stage00_inputs["stage00_runtime_run_dir"],
    "stage00_drive_path_parts": stage00_inputs["stage00_drive_path_parts"],
    "stage02_families": stage01_config["stage02_handoff"]["recommended_model_families"],
    "no_final_model_selected": stage01_config["selection_rules"]["no_final_model_selected"],
    "holdout_test_contact": stage01_config["holdout_test_contact"],
}, indent=2))


## Stage 00 Input Check

Stage 01 requires frozen Stage 00 artifacts from one exact run folder.
When `RUN_STAGE01=True`, this cell first checks
`/content/lst_models_results/00_data_split_label_freeze/<run_id>/`.
If files are missing and `RUN_STAGE00_DRIVE_SYNC=True`, it downloads the
required files from the exact Google Drive path parts in the config.

This cell does not scan for a latest run, does not use stale `/content/...`
inventory paths, and does not read or summarize protected future rows.


In [ ]:
from lst_models.artifacts import require_artifacts


def quote_drive_query_value(value):
    return str(value).replace("\\", "\\\\").replace("'", "\\'")


def find_unique_drive_child(service, parent_id, name, mime_type=None):
    escaped_name = quote_drive_query_value(name)
    query_parts = [
        f"name = '{escaped_name}'",
        f"'{parent_id}' in parents",
        "trashed = false",
    ]
    if mime_type:
        query_parts.append(f"mimeType = '{mime_type}'")
    query = " and ".join(query_parts)
    response = service.files().list(
        q=query,
        spaces="drive",
        fields="files(id, name, mimeType, size)",
        pageSize=10,
    ).execute()
    matches = response.get("files", [])
    if len(matches) != 1:
        raise FileNotFoundError(
            f"expected exactly one Drive item named {name!r} under parent {parent_id}; "
            f"found {len(matches)}"
        )
    return matches[0]


def resolve_drive_folder(service, path_parts):
    folder_id = "root"
    folder_mime = "application/vnd.google-apps.folder"
    for folder_name in path_parts:
        folder = find_unique_drive_child(service, folder_id, folder_name, folder_mime)
        folder_id = folder["id"]
    return folder_id


def download_drive_file(service, file_id, output_path):
    from googleapiclient.http import MediaIoBaseDownload

    output_path.parent.mkdir(parents=True, exist_ok=True)
    request = service.files().get_media(fileId=file_id)
    with output_path.open("wb") as handle:
        downloader = MediaIoBaseDownload(handle, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()


def sync_stage00_artifacts_from_drive(required_names):
    try:
        from google.colab import auth
        from googleapiclient.discovery import build
    except ImportError as exc:
        raise RuntimeError(
            "Stage 00 Drive sync only works inside Colab with Google API dependencies. "
            "Alternatively, place the frozen Stage 00 run folder at "
            f"{STAGE00_OUTPUT_DIR} before setting RUN_STAGE01=True."
        ) from exc

    auth.authenticate_user()
    service = build("drive", "v3")
    run_folder_id = resolve_drive_folder(service, STAGE00_DRIVE_PATH_PARTS)
    for artifact_name in required_names:
        output_path = STAGE00_OUTPUT_DIR / artifact_name
        if output_path.exists():
            continue
        metadata = find_unique_drive_child(service, run_folder_id, artifact_name)
        download_drive_file(service, metadata["id"], output_path)
        print(f"Downloaded Stage 00 artifact: {output_path}")


required_stage00_artifacts = stage01_config["inputs"]["required_stage00_artifacts"]

if RUN_STAGE01:
    missing_before = [name for name in required_stage00_artifacts if not (STAGE00_OUTPUT_DIR / name).exists()]
    if missing_before and RUN_STAGE00_DRIVE_SYNC:
        print("Missing Stage 00 artifacts in runtime; syncing exact frozen run from Drive.")
        print("Drive path parts:", STAGE00_DRIVE_PATH_PARTS)
        sync_stage00_artifacts_from_drive(missing_before)

    stage00_paths = require_artifacts(STAGE00_OUTPUT_DIR, required_stage00_artifacts)
    print("Stage 00 artifact presence check passed.")
    print(json.dumps({name: str(path) for name, path in stage00_paths.items()}, indent=2))
else:
    print("RUN_STAGE01=False; Stage 00 artifact check not executed.")


## Run Stage 01

The package-backed stage entry point is expected at
`lst_models.stages.feature_window_search.run_stage`. The import and
run are both guarded so the committed notebook remains inert by
default.


In [ ]:
if RUN_STAGE01:
    try:
        from lst_models.stages.feature_window_search import run_stage
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "Missing Stage 01 package entry point: "
            "src/lst_models/stages/feature_window_search.py. "
            "Implement package logic before setting RUN_STAGE01=True."
        ) from exc

    result = run_stage(stage01_config)
    display(result)
else:
    print("RUN_STAGE01=False; Stage 01 feature/window screening not executed.")


## Artifact Display

After an approved run, display only Stage 01 artifacts.
Interpretation must use Stage 01 wording: candidate inputs and
model-family shortlist, not final model selection.


In [ ]:
if RUN_ARTIFACT_DISPLAY:
    import pandas as pd

    if "result" not in globals():
        raise RuntimeError(
            "RUN_ARTIFACT_DISPLAY=True requires running Stage 01 in this runtime first. "
            "The display cell uses result.output_dir so it reads the exact run folder, "
            "not the Stage 01 parent directory or a latest-run scan."
        )

    run_dir = Path(result.output_dir)
    inventory_path = run_dir / "artifact_inventory.csv"
    summary_path = run_dir / "01_feature_window_search_summary.csv"
    candidate_path = run_dir / "01_candidate_inputs.json"
    ledger_path = run_dir / "01_train_inner_probe_ledger.csv"

    for path in [inventory_path, summary_path, candidate_path, ledger_path]:
        require_path(path)

    inventory = pd.read_csv(inventory_path)
    summary = pd.read_csv(summary_path)
    ledger = pd.read_csv(ledger_path)
    with candidate_path.open("r", encoding="utf-8") as handle:
        candidate_inputs = json.load(handle)

    assert candidate_inputs["no_final_model_selected"] is True
    assert candidate_inputs["holdout_test_contact"] is False

    display(inventory)
    display(summary)
    display(ledger.head(20))
    print(json.dumps(candidate_inputs, indent=2))
else:
    print("RUN_ARTIFACT_DISPLAY=False; no Stage 01 artifacts displayed.")


## Interpretation Guard

Allowed wording after Stage 01:

- selected candidate inputs for Stage 02 train-inner HPO
- shortlisted model families for Stage 02
- no final model selected in Stage 01

Do not claim a final model, validation winner, holdout winner, or
proof that one model family beats another from this notebook.
